# Báo Cáo Cuối Kỳ - K-Means Image Segmentation

Notebook này trình bày lại project theo phong cách các lab trước: nêu bài toán, nguồn dữ liệu, workflow, cân bằng dữ liệu, training, so sánh mô hình, hình ảnh sau training và phần review cuối. Notebook chỉ dùng Markdown và hình ảnh, không chứa code cell.

## 1. Define Problem - Xác định bài toán

Mục tiêu của bài là phân đoạn một ảnh thành `K` cụm bằng thuật toán K-Means. Với mỗi pixel, chương trình dùng đặc trưng màu để gán pixel vào centroid gần nhất, sau đó thay màu pixel bằng màu centroid của cụm.

Project chọn ảnh phong cảnh vì các vùng như trời, núi, cây, nước, cát và mây thường có khác biệt màu rõ ràng, phù hợp với bài toán phân cụm không giám sát. “Độ chính xác” trong report này được hiểu là chất lượng phân cụm nội bộ và kiểm tra trực quan, không phải IoU/Dice vì đề bài không cung cấp ground-truth mask.

## 2. Assignment Mapping - Đối chiếu yêu cầu đề bài

| Yêu cầu đề bài | Phần đã thực hiện |
|---|---|
| Load the image | Đọc ảnh bằng PIL trong `src/image_io.py` |
| Convert color space | So sánh `RGB`, `HSV`, `LAB` |
| Resize image | Resize về `max_side=128` để train nhanh và ổn định |
| Flatten pixels | Biến ảnh thành ma trận pixel feature |
| Apply K-Means | Cài đặt K-Means from scratch bằng NumPy |
| Assign nearest centroid | Gán pixel theo Euclidean distance |
| Update centroids | Cập nhật centroid bằng mean của pixel trong cụm |
| Repeat until convergence | Dừng bằng `max_iter` và `tol` |
| Segment image | Thay màu pixel bằng màu centroid |
| Visualize results | Xuất segmented image, comparison image và K-grid |

## 3. Workflow tổng quát và dẫn chứng đã làm

1. Thu thập data thô: manifest `../data/manifest/raw_landscape_manifest.csv` và ảnh trong `../data/raw/api`.
2. Làm sạch và cân bằng dữ liệu: clean manifest `../data/manifest/clean_landscape_manifest.csv` và báo cáo `../reports/metrics/data_balance_report.json`.
3. Tiền xử lý: đọc ảnh RGB, resize, đổi color space, flatten pixel và tùy chọn thêm tọa độ `(x, y)`.
4. Train K-Means: chạy `K=2..10`, `rgb/hsv/lab`, `use_xy=false/true`.
5. Xuất output sau training: ảnh trong `../reports/figures`, labels trong `../data/labels`, model artifacts trong `../models`.
6. So sánh và review: metrics `../reports/metrics/model_comparison.csv`, best model `../reports/metrics/best_model_by_image.json`, source review `../reports/metrics/source_review.json`.

## 4. Kiểm tra và cân bằng dữ liệu

Hệ thống audit từng ảnh theo width, height, aspect ratio, mean intensity, standard deviation, file size, source và query. Các ảnh lệch quá xa phân phối được đánh dấu bằng IQR và z-score trước khi chọn tập balanced.

**Tổng quan sau cân bằng**

| raw_images | clean_images | real_selected | augmentation_selected | max_augmentation | model_runs |
| --- | --- | --- | --- | --- | --- |
| 22 | 20 | 20 | 0 | 8 | 1080 |

**Khoảng giá trị chính**

| Feature | Min | Mean | Max |
|---|---:|---:|---:|
| width | 720 | 2848.90 | 7580 |
| height | 663 | 1899.55 | 4667 |
| mean intensity | 65.87 | 107.26 | 164.67 |
| std intensity | 44.59 | 57.10 | 77.39 |

![Balanced dataset source distribution](../reports/figures/report_source_distribution.png)

*Balanced dataset source distribution*

## 5. Dataset sau lọc và đánh nhãn

Dataset cuối cùng giữ **20 ảnh clean**, trong đó có **20 ảnh real** và **0 ảnh augmentation**. Nhãn metadata vẫn là `dataset_label=landscape` và `image_domain=landscape`. Nhãn pixel-level là cluster label sinh ra sau K-Means, không dùng mask giám sát.

**Phân phối source**

| source | count |
| --- | --- |
| wikimedia_commons | 4 |
| wikimedia_commons_category | 2 |
| wikimedia_existing_raw | 14 |

**Phân phối query**

| query | count |
| --- | --- |
| Desert landscapes | 1 |
| Landscape photographs | 1 |
| beach landscape | 1 |
| existing raw landscape | 14 |
| forest landscape | 1 |
| lake landscape | 1 |
| national park | 1 |

## 6. Preprocessing

Mỗi ảnh được resize về `max_side=128`, chuyển sang `RGB`, `HSV` hoặc `LAB`, sau đó flatten thành vector pixel. Với chế độ spatial, vector pixel được mở rộng thêm tọa độ chuẩn hóa `(x, y)` để segmentation ổn định hơn ở các vùng ảnh gần nhau.

Project không dùng supervised mask, object detection, U-Net, SAM, Mask R-CNN hoặc mô hình deep learning. Phạm vi vẫn đúng với K-Means Image Segmentation.

## 7. Training Models

| Thành phần | Giá trị |
|---|---|
| Số ảnh clean | 20 |
| K values | [2, 3, 4, 5, 6, 7, 8, 9, 10] |
| Color spaces | ['rgb', 'hsv', 'lab'] |
| Spatial modes | [False, True] |
| Tổng số model runs | 1080 |

Mỗi model run tạo đầy đủ segmented image, side-by-side comparison image, pixel-label `.npy`, và model artifact `.npz`.

## 8. So sánh mô hình

Vì bài toán không có ground-truth mask, mô hình được so sánh bằng internal unsupervised metrics và visual review. Report không giải thích từng giá trị `K` riêng lẻ; thay vào đó dùng ranking tổng hợp và các biểu đồ đại diện.

| Metric | Hướng tốt hơn | Ý nghĩa |
|---|---|---|
| silhouette_sample | cao hơn | Cụm tách nhau rõ hơn |
| davies_bouldin_sample | thấp hơn | Cụm gọn và ít chồng lấn hơn |
| calinski_harabasz_sample | cao hơn | Cụm tách biệt tốt hơn |
| inertia_per_pixel | thấp hơn | Pixel gần centroid hơn |
| cluster_balance | cao hơn | Tránh cụm quá nhỏ hoặc collapsed |
| ranking_score | thấp hơn | Điểm tổng hợp để chọn model |

![Average ranking score by K and color space](../reports/figures/report_ranking_score_by_k.png)

*Average ranking score by K and color space*

![Average ranking score by color space](../reports/figures/report_color_space_comparison.png)

*Average ranking score by color space*

**Top 12 model runs theo ranking score**

| image_id | k | color_space | use_xy | silhouette_sample | davies_bouldin_sample | inertia_per_pixel | cluster_balance | ranking_score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | 0.7680300230392144 | 0.3889383730290883 | 3397.796426673979 | 0.9824740167108212 | 1.4515519988570056 |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | True | 0.7585101716545529 | 0.397228702572854 | 3467.828345595299 | 0.9824740167108212 | 1.4714017305432043 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | 0.7768994629167776 | 0.2886640635269787 | 766.5654205675322 | 0.648 | 1.5280869720455237 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | True | 0.7543469492889678 | 0.3209496396153748 | 838.4400671829987 | 0.6494432628549981 | 1.5904329357596856 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | 0.663717964580371 | 0.450641220531663 | 2001.840447158684 | 0.9514781917009149 | 1.5956772318061554 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | False | 0.6591137989567841 | 0.4640409341143357 | 1104.0615972266949 | 0.885970531710442 | 1.619722239096586 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | 0.6615684530453798 | 0.4770289328753115 | 2524.9980681173874 | 0.9713716252944374 | 1.6275280083465953 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | True | 0.6525974237422562 | 0.4670227509144705 | 2090.7033874196254 | 0.9507023588656242 | 1.6282963713982146 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | True | 0.6512798551101667 | 0.4917372826275049 | 2608.9041932243795 | 0.9731592310482408 | 1.6540050756757108 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | lab | True | 0.635919186911147 | 0.5002377159598438 | 1192.7235343260672 | 0.8852459016393442 | 1.6875558995444644 |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | 0.6424904349545513 | 0.464546509915471 | 985.383811400412 | 0.7890724269377383 | 1.7223882020147208 |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | 0.6614702077130843 | 0.4464239423673495 | 2420.6202740715503 | 0.814959234314073 | 1.7492137133254575 |

## 9. Đánh giá mô hình tốt nhất

Với mỗi ảnh, model tốt nhất là cấu hình có `ranking_score` thấp nhất. Bảng dưới đây là cấu hình best-model theo từng ảnh.

![Best-model K distribution](../reports/figures/report_best_k_distribution.png)

*Best-model K distribution*

| image_id | k | color_space | use_xy | silhouette_sample | davies_bouldin_sample | ranking_score |
| --- | --- | --- | --- | --- | --- | --- |
| Catoctin_Mountain_and_farm_MD1_jpg | 2 | lab | False | 0.7680300230392144 | 0.3889383730290883 | 1.4515519988570056 |
| Sunset_by_Caspar_David_Friedrich_jpg | 2 | rgb | False | 0.7768994629167776 | 0.2886640635269787 | 1.5280869720455237 |
| C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | 2 | rgb | False | 0.663717964580371 | 0.450641220531663 | 1.5956772318061554 |
| Barn_on_Mastl_mountain_Gherd_ina_jpg | 2 | rgb | False | 0.6615684530453798 | 0.4770289328753115 | 1.6275280083465953 |
| Algoma_Gabrielle_Rock_I0012362_jpg | 2 | lab | False | 0.6424904349545513 | 0.464546509915471 | 1.7223882020147208 |
| Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | 2 | rgb | False | 0.6614702077130843 | 0.4464239423673495 | 1.7492137133254575 |
| Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | 2 | rgb | False | 0.6463516030822222 | 0.4776364985860166 | 1.7574422896030486 |
| Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | 2 | lab | False | 0.7852136269873675 | 0.3137703042279379 | 1.7575181319128204 |
| Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | 2 | rgb | False | 0.6245320484158806 | 0.5304142507342385 | 1.7757037836846037 |
| Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | 3 | lab | False | 0.875077260570786 | 0.2084735871637714 | 1.8488095325818388 |
| Ruisseau_du_Vialais_-_March_2021_-_B_jpg | 3 | lab | False | 0.8390200224812907 | 0.2337744664684616 | 1.8512513132609527 |
| Bontecou_Lake_Milky_Way_panorama_jpg | 3 | lab | False | 0.8359389243991842 | 0.2633130697616206 | 1.978389245044346 |
| Castle_Mountain_jpg | 2 | lab | False | 0.6381320390338326 | 0.6136415221891759 | 2.072480937059499 |
| Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | 2 | rgb | False | 0.5400444062908873 | 0.6752301069700718 | 2.08084331621532 |
| jpg | 2 | lab | False | 0.6729276680462352 | 0.5878205322606979 | 2.101153243700912 |
| Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | 2 | rgb | False | 0.7549214309460072 | 0.3563897037397857 | 2.105833867868676 |
| 004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | 2 | rgb | False | 0.5537416068602463 | 0.6251626015634806 | 2.142069657872005 |
| Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | 2 | lab | False | 0.7767430361162948 | 0.2713149258660801 | 2.1538381192350964 |
| Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | 2 | lab | False | 0.6349555459172955 | 0.6678555841204598 | 2.273377819370455 |
| Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | 3 | lab | False | 0.8103970887812879 | 0.3781680639510036 | 2.280080992452206 |

## 10. Hình ảnh sau khi training model

Các hình dưới đây là output comparison sau training, gồm ảnh gốc và ảnh đã segment đặt cạnh nhau. Report chỉ đưa best-model gallery để notebook gọn và dễ review; toàn bộ 1080 ảnh output vẫn nằm trong `../reports/figures`.

![Catoctin_Mountain_and_farm_MD1_jpg | K=2 | LAB | use_xy=False | ranking_score=1.4516](../reports/figures/Catoctin_Mountain_and_farm_MD1_jpg_k2_lab_color_comparison.png)

*Catoctin_Mountain_and_farm_MD1_jpg | K=2 | LAB | use_xy=False | ranking_score=1.4516*

![Sunset_by_Caspar_David_Friedrich_jpg | K=2 | RGB | use_xy=False | ranking_score=1.5281](../reports/figures/Sunset_by_Caspar_David_Friedrich_jpg_k2_rgb_color_comparison.png)

*Sunset_by_Caspar_David_Friedrich_jpg | K=2 | RGB | use_xy=False | ranking_score=1.5281*

![C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | K=2 | RGB | use_xy=False | ranking_score=1.5957](../reports/figures/C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg_k2_rgb_color_comparison.png)

*C_sar_van_Loo_1743-1821_-_Sunset_Landscape_-_609014_-_National_Trust_jpg | K=2 | RGB | use_xy=False | ranking_score=1.5957*

![Barn_on_Mastl_mountain_Gherd_ina_jpg | K=2 | RGB | use_xy=False | ranking_score=1.6275](../reports/figures/Barn_on_Mastl_mountain_Gherd_ina_jpg_k2_rgb_color_comparison.png)

*Barn_on_Mastl_mountain_Gherd_ina_jpg | K=2 | RGB | use_xy=False | ranking_score=1.6275*

![Algoma_Gabrielle_Rock_I0012362_jpg | K=2 | LAB | use_xy=False | ranking_score=1.7224](../reports/figures/Algoma_Gabrielle_Rock_I0012362_jpg_k2_lab_color_comparison.png)

*Algoma_Gabrielle_Rock_I0012362_jpg | K=2 | LAB | use_xy=False | ranking_score=1.7224*

![Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | K=2 | RGB | use_xy=False | ranking_score=1.7492](../reports/figures/Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos_k2_rgb_color_comparison.png)

*Pirogue_and_boat_on_the_Mekong_with_colorful_sky_at_sunset_in_Luang_Prabang_Laos | K=2 | RGB | use_xy=False | ranking_score=1.7492*

![Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | K=2 | RGB | use_xy=False | ranking_score=1.7574](../reports/figures/Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum__k2_rgb_color_comparison.png)

*Carl_Spitzweg_1808-1885_-_Mountain_Landscape_-_WA1954_70_164_-_Ashmolean_Museum_ | K=2 | RGB | use_xy=False | ranking_score=1.7574*

![Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | K=2 | LAB | use_xy=False | ranking_score=1.7575](../reports/figures/Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg_k2_lab_color_comparison.png)

*Shishkin_Ivan_-_Morning_in_a_Pine_Forest_jpg | K=2 | LAB | use_xy=False | ranking_score=1.7575*

![Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | K=2 | RGB | use_xy=False | ranking_score=1.7757](../reports/figures/Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg_k2_rgb_color_comparison.png)

*Gillis_van_Coninxloo_-_Forest_Landscape_-_38_70_-_Detroit_Institute_of_Arts_jpg | K=2 | RGB | use_xy=False | ranking_score=1.7757*

![Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | K=3 | LAB | use_xy=False | ranking_score=1.8488](../reports/figures/Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg_k3_lab_color_comparison.png)

*Parasols_Evening_Beach_Rincon_de_la_Victoria_Andalusia_Spain_jpg | K=3 | LAB | use_xy=False | ranking_score=1.8488*

![Ruisseau_du_Vialais_-_March_2021_-_B_jpg | K=3 | LAB | use_xy=False | ranking_score=1.8513](../reports/figures/Ruisseau_du_Vialais_-_March_2021_-_B_jpg_k3_lab_color_comparison.png)

*Ruisseau_du_Vialais_-_March_2021_-_B_jpg | K=3 | LAB | use_xy=False | ranking_score=1.8513*

![Bontecou_Lake_Milky_Way_panorama_jpg | K=3 | LAB | use_xy=False | ranking_score=1.9784](../reports/figures/Bontecou_Lake_Milky_Way_panorama_jpg_k3_lab_color_comparison.png)

*Bontecou_Lake_Milky_Way_panorama_jpg | K=3 | LAB | use_xy=False | ranking_score=1.9784*

![Castle_Mountain_jpg | K=2 | LAB | use_xy=False | ranking_score=2.0725](../reports/figures/Castle_Mountain_jpg_k2_lab_color_comparison.png)

*Castle_Mountain_jpg | K=2 | LAB | use_xy=False | ranking_score=2.0725*

![Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | K=2 | RGB | use_xy=False | ranking_score=2.0808](../reports/figures/Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene_k2_rgb_color_comparison.png)

*Painting_of_Maharaja_Ranjit_Singh_seated_and_wearing_red_robe_with_natural_scene | K=2 | RGB | use_xy=False | ranking_score=2.0808*

![jpg | K=2 | LAB | use_xy=False | ranking_score=2.1012](../reports/figures/jpg_k2_lab_color_comparison.png)

*jpg | K=2 | LAB | use_xy=False | ranking_score=2.1012*

![Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | K=2 | RGB | use_xy=False | ranking_score=2.1058](../reports/figures/Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg_k2_rgb_color_comparison.png)

*Gyps_rueppellii_-Nairobi_National_Park_Kenya-8-4c_jpg | K=2 | RGB | use_xy=False | ranking_score=2.1058*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | K=2 | RGB | use_xy=False | ranking_score=2.1421](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k2_rgb_color_comparison.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg | K=2 | RGB | use_xy=False | ranking_score=2.1421*

![Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | K=2 | LAB | use_xy=False | ranking_score=2.1538](../reports/figures/Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J_k2_lab_color_comparison.png)

*Wooden_staircase_steps_in_the_forest_of_Hallasan_Park_Eorimok_Trail_at_dusk_on_J | K=2 | LAB | use_xy=False | ranking_score=2.1538*

![Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | K=2 | LAB | use_xy=False | ranking_score=2.2734](../reports/figures/Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg_k2_lab_color_comparison.png)

*Beech_Forest_AU_Great_Otway_National_Park_Beauchamp_Falls_--_2019_--_1271_jpg | K=2 | LAB | use_xy=False | ranking_score=2.2734*

![Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | K=3 | LAB | use_xy=False | ranking_score=2.2801](../reports/figures/Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png_k3_lab_color_comparison.png)

*Wind_Mountain_Columbia_R_-_NARA_-_102278851_page_1_png | K=3 | LAB | use_xy=False | ranking_score=2.2801*

## 11. K-grid highlights

K-grid giúp kiểm tra trực quan việc tăng `K` ảnh hưởng đến segmentation như thế nào. Phần này chỉ chọn một số hình đại diện, không giải thích từng `K`.

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_color.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_hsv_xy.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_color.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_lab_xy.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_color.png*

![004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png](../reports/figures/004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png)

*004_Chital_in_Keoladeo_National_Park_Photo_by_Giles_Laurent_jpg_k_grid_rgb_xy.png*

## 12. Giao diện chương trình

Giao diện Python dùng Streamlit trong `app.py`. Người dùng có thể upload ảnh, chọn `K`, chọn color space, bật/tắt `use_xy`, sau đó xem ảnh segmentation và metrics sau khi chạy.

## 13. Review notebook và source code

| Check | Result |
|---|---|
| review status | passed |
| clean images | 20 |
| model runs | 1080 |
| expected model runs | 1080 |
| notebook code cells | 0 |
| assignment alignment | The source implements unsupervised K-Means Image Segmentation with landscape images. |

Notebook được tạo lại dưới dạng markdown-only, dùng UTF-8, dùng Markdown image syntax chuẩn và chỉ nhúng hình output sau training theo hướng tổng hợp. Source review kiểm tra manifest, metrics, labels, model artifacts, hình ảnh output và alignment với đề bài.

## 14. Kết luận

Chương trình đã hoàn thành đúng trọng tâm đề bài: load ảnh, tiền xử lý màu, flatten pixel, train K-Means, gán cụm, cập nhật centroid, segment ảnh và trực quan hóa kết quả. Dataset đã được cân bằng lại trước khi train, model được so sánh bằng metrics không giám sát, và notebook cuối cùng trình bày kết quả theo phong cách report lab thay vì liệt kê toàn bộ từng cấu hình `K`.